# 📘 Módulo 7 — Estatística para Machine Learning
## Fundamentos Preditivos e Generalização de Modelos

- 🔵 Conteúdo essencial para conectar Estatística → Machine Learning
- 🟣 Conteúdo expandido (Livro Didático)
- 🟠 Conteúdo avançado (opcional)

Este módulo marca a transição entre:

**Inferência Estatística (módulos 4–6)**  
e  
**Aprendizado de Máquina (módulos 8–10)**.

Aqui você aprenderá os pilares que permitem que modelos estatísticos
se tornem modelos preditivos capazes de generalizar para novos dados.

---

# 🎯 Objetivos do Módulo 7

Ao final deste módulo, você será capaz de:

- entender a diferença entre **ajuste** e **generalização**  
- dividir dados em **treino** e **teste**  
- medir erro preditivo com RMSE, MAE e R² preditivo  
- entender **overfitting** e **underfitting**  
- aplicar **validação cruzada**  
- usar **regularização** (Ridge e Lasso)  
- comparar modelos e escolher o melhor  
- construir um **modelo preditivo completo**  

Este módulo é a ponte direta para Machine Learning clássico.

# 📚 Índice

0. [Setup — bibliotecas e dados](#setup)
1. [Introdução: Estatística → Machine Learning](#intro)
2. [Treino e Teste](#train_test)
3. [Métricas de Erro Preditivo](#metricas)
4. [Overfitting e Underfitting](#over_under)
5. [Validação Cruzada](#cv)
6. [Regularização: Ridge e Lasso](#regularizacao)
7. [Seleção de Modelos](#selecao)
8. [Projeto Final — Modelo Preditivo Completo](#projeto)
9. [Apêndice Matemático Avançado](#apendice)

<a id="setup"></a>
# 0. Setup — bibliotecas e dados

Usaremos o mesmo dataset *teaching ratings* dos módulos anteriores,
agora com foco em **previsão**, não apenas inferência.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
ratings_df = pd.read_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv")
ratings_df.head()

---
# 🔵 Conexão com os módulos anteriores

Nos módulos 4, 5 e 6, você aprendeu:

- testes de hipótese  
- regressão simples  
- regressão múltipla  
- regressão com dummies  
- regressão como alternativa a T‑test, ANOVA e Pearson  

Esses métodos respondem perguntas como:

- “Existe diferença?”  
- “Existe relação?”  
- “Qual é o efeito de X sobre Y?”  

Agora, no Módulo 7, mudamos a pergunta para:

> **“Quão bem conseguimos prever Y para novos dados?”**

Essa mudança é o coração do Machine Learning.

<a id="intro"></a>
# 1. Introdução: Estatística → Machine Learning

A regressão que você aprendeu no módulo 6 pode ser usada de duas formas:

## ✔️ 1. Inferência (estatística)
- foco em significância  
- valor‑p  
- testes t e F  
- interpretação de coeficientes  

## ✔️ 2. Predição (machine learning)
- foco em erro preditivo  
- treino/teste  
- RMSE, MAE, R² preditivo  
- generalização  
- regularização  

A fórmula é a mesma:

$$
\hat{Y} = \beta_0 + \beta_1 X_1 + \dots + \beta_k X_k
$$

O que muda é **o objetivo**.

---
# 🧠 Por que precisamos de treino e teste?

Porque um modelo pode:

- **memorizar** os dados (overfitting)  
- **generalizar** para novos dados (bom modelo)  

Machine Learning é sobre **generalização**.

Para medir isso, precisamos separar:

- dados usados para ajustar o modelo (**treino**)  
- dados nunca vistos pelo modelo (**teste**)  

Isso será a base das próximas partes.

<details>
<summary>🟠 Explicação avançada — O problema da generalização</summary>

Um modelo pode ter:

- erro baixo no treino  
- erro alto no teste  

Isso significa que ele aprendeu **ruído**, não **padrão**.

Formalmente:

$$
\text{Erro total} = \text{Viés}^2 + \text{Variância} + \text{Ruído}
$$

Machine Learning é sobre equilibrar viés e variância.

</details>

<a id="train_test"></a>
# 2. Treino e Teste — A Base do Machine Learning Supervisionado

Nos módulos anteriores, usamos **todos os dados** para ajustar modelos.
Isso funciona para inferência estatística, mas NÃO funciona para predição.

Em Machine Learning, o objetivo é:

> **Avaliar o desempenho do modelo em dados que ele nunca viu.**

Para isso, dividimos o dataset em:

- **treino (train)** → usado para ajustar o modelo  
- **teste (test)** → usado para avaliar generalização  

Essa é a mudança conceitual mais importante entre estatística e ML.

---
# 2.1 Por que dividir em treino e teste?

Porque um modelo pode:

- aprender padrões reais (**bom**)  
- aprender ruído (**ruim**)  

Quando o modelo aprende ruído, chamamos isso de **overfitting**.

Para detectar overfitting, precisamos de dados que o modelo **não viu**.

Por isso dividimos o dataset.

---
# 2.2 Criando treino e teste com scikit‑learn

Vamos prever `eval` usando `beauty` como exemplo simples.

In [ ]:
X = ratings_df[["beauty"]]
y = ratings_df["eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

len(X_train), len(X_test)

🟣 **Interpretação**

- 75% dos dados → treino  
- 25% dos dados → teste  

O modelo só verá os dados de treino.

---
# 2.3 Ajustando o modelo no conjunto de treino

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

model.coef_, model.intercept_

🟣 **Interpretação**

- coeficiente β₁  
- intercepto β₀  

Esses valores foram aprendidos **somente com os dados de treino**.

---
# 2.4 Avaliando o modelo no conjunto de teste

Agora medimos o desempenho em dados **nunca vistos**.

In [ ]:
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

rmse, mae, r2

🟣 **Interpretação das métricas**

- **RMSE** → erro médio em unidades da variável  
- **MAE** → erro absoluto médio  
- **R² preditivo** → quanta variância o modelo explica no teste  

Aqui o R² é baixo, como esperado (beauty explica pouco eval).

---
# 2.5 Visualização: valores reais vs previstos

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel("Valores reais (y_test)")
plt.ylabel("Valores previstos (y_pred)")
plt.title("Predição no conjunto de teste")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Se o modelo fosse perfeito, todos os pontos estariam na diagonal.  
- A dispersão mostra que o modelo tem poder preditivo limitado.  

---
# 2.6 Comparando erro no treino vs erro no teste

Isso é essencial para detectar overfitting.

In [ ]:
y_pred_train = model.predict(X_train)

rmse_train = mean_squared_error(y_train, y_pred_train, squared=False)
rmse_test  = rmse

rmse_train, rmse_test

🟣 **Interpretação**

- Se **RMSE_train << RMSE_test**, temos overfitting.  
- Se **RMSE_train ≈ RMSE_test**, o modelo generaliza bem.  

Aqui os valores são próximos → sem overfitting.

---
# 2.7 O que aprendemos nesta parte?

✔️ Machine Learning exige separar treino e teste  
✔️ O modelo aprende apenas com o treino  
✔️ O teste mede generalização  
✔️ RMSE, MAE e R² preditivo são métricas essenciais  
✔️ Comparar erro de treino e teste detecta overfitting  

A partir daqui, estamos oficialmente no território de ML.

<details>
<summary>🟠 Explicação avançada — Por que treino/teste funciona?</summary>

A ideia vem da teoria de generalização:

- Queremos estimar o erro esperado em novos dados:

$$
E[(Y - \hat{f}(X))^2]
$$

- Mas não temos acesso à distribuição real.
- Então usamos um **proxy**: o conjunto de teste.

O conjunto de teste é uma amostra independente da mesma distribuição.

Por isso ele funciona como estimador do erro real.

</details>

<a id="metricas"></a>
# 3. Métricas de Erro Preditivo

No módulo 6, avaliamos modelos usando:

- valor‑p  
- significância dos coeficientes  
- R² (no conjunto completo)  

Isso é ótimo para **inferência**, mas insuficiente para **predição**.

Em Machine Learning, o foco muda para:

> **Quão bem o modelo prevê novos dados?**

Para isso, usamos métricas de erro preditivo calculadas no **conjunto de teste**.

---
# 3.1 RMSE — Root Mean Squared Error

O RMSE mede o erro médio em unidades da variável Y:

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2}
$$

Interpretação:
- penaliza erros grandes  
- é a métrica mais usada em regressão preditiva  
- quanto menor, melhor  

In [ ]:
rmse = mean_squared_error(y_test, y_pred, squared=False)
rmse

🟣 **Interpretação**

O RMSE indica, em média, o tamanho do erro de previsão.

Exemplo:
- RMSE = 0.5 → o modelo erra ~0.5 pontos na avaliação  

---
# 3.2 MAE — Mean Absolute Error

O MAE mede o erro absoluto médio:

$$
MAE = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|
$$

Interpretação:
- mais robusto a outliers  
- fácil de interpretar  
- quanto menor, melhor  

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mae

🟣 **Interpretação**

O MAE diz, em média, quanto o modelo erra sem penalizar erros grandes.

---
# 3.3 R² preditivo — Coeficiente de Determinação no Teste

O R² tradicional mede:

> proporção da variância explicada **no conjunto de treino**.

Mas o que importa é:

> **Quanto da variância o modelo explica em dados novos?**

Por isso usamos o R² no conjunto de teste:

$$
R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}
$$

In [ ]:
r2 = r2_score(y_test, y_pred)
r2

🟣 **Interpretação**

- R² próximo de 1 → excelente predição  
- R² próximo de 0 → modelo não explica quase nada  
- R² negativo → modelo pior que uma média constante  

Aqui o R² é baixo, como esperado (beauty explica pouco eval).

---
# 3.4 Comparando RMSE, MAE e R²

Cada métrica responde a uma pergunta:

| Métrica | Pergunta respondida |
|--------|----------------------|
| **RMSE** | Quanto o modelo erra, penalizando erros grandes? |
| **MAE** | Quanto o modelo erra, em média? |
| **R²** | Quanto da variância o modelo explica? |

Em ML, usamos **todas juntas**.

---
# 3.5 Visualização: Erro de previsão

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(range(len(y_test)), y_test - y_pred, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.title("Erros de previsão (resíduos no conjunto de teste)")
plt.xlabel("Observação")
plt.ylabel("Erro (y - y_pred)")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- erros próximos de zero → boa previsão  
- dispersão grande → modelo fraco  
- padrões → possível não linearidade  

---
# 3.6 Comparando erro no treino vs teste

Isso é essencial para detectar overfitting.

In [ ]:
rmse_train = mean_squared_error(y_train, model.predict(X_train), squared=False)
rmse_test  = rmse

rmse_train, rmse_test

🟣 **Interpretação**

- **RMSE_train ≪ RMSE_test** → overfitting  
- **RMSE_train ≈ RMSE_test** → boa generalização  

Aqui os valores são próximos → sem overfitting.

---
# 3.7 O que aprendemos nesta parte?

✔️ RMSE mede erro quadrático  
✔️ MAE mede erro absoluto  
✔️ R² preditivo mede variância explicada em dados novos  
✔️ Treino/teste é essencial para medir generalização  
✔️ Comparar erro de treino e teste detecta overfitting  

Agora estamos prontos para o próximo passo:

**Overfitting e Underfitting — o problema central do ML.**

<details>
<summary>🟠 Explicação avançada — Por que RMSE é tão usado?</summary>

O RMSE surge naturalmente da suposição de erros gaussianos:

- minimizar RMSE = maximizar verossimilhança  
- regressão linear OLS = estimador de máxima verossimilhança  

Por isso RMSE é a métrica padrão em regressão.

</details>

<a id="over_under"></a>
# 4. Overfitting e Underfitting — O Problema Central do Machine Learning

Até agora, aprendemos:

- dividir dados em treino e teste  
- ajustar modelos  
- medir erro preditivo  

Agora chegamos ao ponto mais importante:

> **Um modelo pode aprender demais ou aprender de menos.**

Isso define:

- **underfitting** → modelo simples demais  
- **overfitting** → modelo complexo demais  

Machine Learning é o ato de equilibrar esses dois extremos.

---
# 4.1 Underfitting — quando o modelo aprende de menos

Ocorre quando o modelo é **simples demais** para capturar o padrão.

Exemplos:
- usar regressão linear para um padrão curvo  
- usar poucas variáveis  
- usar modelo restrito demais  

Sinais de underfitting:

- erro alto no treino  
- erro alto no teste  
- R² baixo  

Visualmente:

```
Dados: curvos
Modelo: linha reta
```

---
# 4.2 Overfitting — quando o modelo aprende demais

Ocorre quando o modelo:

- aprende ruído  
- memoriza os dados  
- perde capacidade de generalização  

Sinais de overfitting:

- erro muito baixo no treino  
- erro alto no teste  
- modelo instável  

Visualmente:

```
Dados: curvos
Modelo: linha que passa por todos os pontos (zig‑zag)
```

---
# 4.3 Demonstração prática com polinômios

Vamos criar modelos de diferentes complexidades:

- grau 1 → linear (simples)  
- grau 5 → moderado  
- grau 15 → extremamente complexo (overfitting garantido)  

Usaremos `beauty` para prever `eval`.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

def fit_poly(degree):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly  = poly.transform(X_test)

    model_poly = LinearRegression()
    model_poly.fit(X_train_poly, y_train)

    y_pred_train = model_poly.predict(X_train_poly)
    y_pred_test  = model_poly.predict(X_test_poly)

    rmse_train = mean_squared_error(y_train, y_pred_train, squared=False)
    rmse_test  = mean_squared_error(y_test, y_pred_test, squared=False)

    return rmse_train, rmse_test

results = {
    d: fit_poly(d) for d in [1, 5, 15]
}

results

🟣 **Interpretação**

- grau 1 → erros parecidos → sem overfitting  
- grau 5 → erro de treino cai, erro de teste sobe um pouco  
- grau 15 → erro de treino despenca, erro de teste explode  

Isso é **overfitting clássico**.

---
# 4.4 Visualização do overfitting

In [ ]:
degrees = [1, 5, 15]
plt.figure(figsize=(8, 5))

for d in degrees:
    rmse_train, rmse_test = results[d]
    plt.scatter(d, rmse_train, color="blue", label="Treino" if d == 1 else "")
    plt.scatter(d, rmse_test,  color="red",  label="Teste"  if d == 1 else "")

plt.legend()
plt.xlabel("Grau do polinômio")
plt.ylabel("RMSE")
plt.title("Overfitting: erro de treino vs teste")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- erro de treino cai sempre → modelo fica mais complexo  
- erro de teste tem formato de U → mínimo ideal  

Esse gráfico é um dos mais importantes de todo ML.

---
# 4.5 A Curva de Viés–Variância

O erro total pode ser decomposto em:

$$
\text{Erro} = \text{Viés}^2 + \text{Variância} + \text{Ruído}
$$

- **Viés alto** → underfitting  
- **Variância alta** → overfitting  

Machine Learning é sobre equilibrar esses dois.

---
# 4.6 Como evitar overfitting?

Existem 4 estratégias principais:

## ✔️ 1. Dividir em treino e teste  
(já fizemos)

## ✔️ 2. Validação cruzada  
(próxima parte)

## ✔️ 3. Regularização (Ridge e Lasso)  
(parte 6)

## ✔️ 4. Reduzir complexidade  
- menos features  
- modelos mais simples  
- poda em árvores  

Tudo isso será explorado neste módulo.

---
# 4.7 O que aprendemos nesta parte?

✔️ Underfitting = modelo simples demais  
✔️ Overfitting = modelo complexo demais  
✔️ Erro de treino baixo ≠ modelo bom  
✔️ O que importa é o erro no teste  
✔️ Overfitting é o problema central do ML  
✔️ Regularização e validação cruzada são as soluções  

Agora estamos prontos para a próxima parte:

**Validação Cruzada — a forma mais robusta de medir generalização.**

<details>
<summary>🟠 Explicação avançada — Por que modelos complexos overfitam?</summary>

Porque modelos complexos têm **alta capacidade**.

Capacidade = número de funções que o modelo consegue representar.

- Regressão linear → baixa capacidade  
- Polinômio grau 15 → capacidade enorme  
- Redes neurais → capacidade gigantesca  

Quanto maior a capacidade, maior o risco de memorizar ruído.

</details>

<a id="cv"></a>
# 5. Validação Cruzada — O Padrão‑Ouro da Avaliação de Modelos

Até agora, usamos uma única divisão:

- 75% treino  
- 25% teste  

Isso funciona, mas tem um problema:

> **O resultado depende da forma como os dados foram divididos.**

Se a divisão for “sortuda”, o modelo parece ótimo.  
Se for “azarada”, o modelo parece ruim.

Para resolver isso, usamos **validação cruzada (cross‑validation)**.

---
# 5.1 O que é validação cruzada?

A ideia é simples:

1. Dividimos os dados em **K partes** (folds).  
2. Treinamos o modelo em K‑1 partes.  
3. Testamos na parte restante.  
4. Repetimos K vezes.  
5. Fazemos a média do erro.

O método mais comum é o **K‑Fold Cross‑Validation**.

---
# 5.2 K‑Fold Cross‑Validation na prática

Vamos usar K = 5 (padrão mais comum).

In [ ]:
X = ratings_df[["beauty"]]
y = ratings_df["eval"]

model = LinearRegression()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model, X, y,
    scoring="neg_root_mean_squared_error",
    cv=kf
)

scores

🟣 **Interpretação**

- Cada valor é o RMSE (negativo, por convenção do sklearn).  
- Quanto mais próximo de zero, melhor.  
- A média é a estimativa mais robusta do erro preditivo.

In [ ]:
rmse_cv = -scores.mean()
rmse_cv

🟣 **Conclusão**

- Este RMSE é mais confiável do que o RMSE de uma única divisão.  
- Ele reduz variância e evita conclusões enganosas.  

---
# 5.3 Comparando treino/teste vs validação cruzada

Vamos comparar:

In [ ]:
rmse_train_test = mean_squared_error(y_test, y_pred, squared=False)
rmse_cv

🟣 **Interpretação**

- RMSE treino/teste → depende da divisão  
- RMSE CV → média de 5 divisões diferentes  

A validação cruzada é **mais estável e mais justa**.

---
# 5.4 Visualização dos erros por fold

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, 6), -scores, marker="o")
plt.xlabel("Fold")
plt.ylabel("RMSE")
plt.title("Erro por fold na validação cruzada")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Se os erros variam muito → modelo instável  
- Se os erros são parecidos → modelo robusto  

Aqui a variação é pequena → bom sinal.

---
# 5.5 Por que validação cruzada é tão importante?

Porque ela:

✔️ usa todos os dados para treino  
✔️ usa todos os dados para teste  
✔️ reduz variância da avaliação  
✔️ evita conclusões enganosas  
✔️ permite comparar modelos de forma justa  
✔️ é o padrão‑ouro em ML clássico  

Em competições (Kaggle, etc.), CV é obrigatório.

---
# 5.6 Quando usar validação cruzada?

Use CV quando:

- dataset é pequeno  
- você quer comparar modelos  
- você quer evitar overfitting  
- você quer estimar erro real com precisão  

Não use CV quando:

- dataset é gigantesco (milhões de linhas)  
- o custo computacional é muito alto  

Mas para datasets típicos, CV é a melhor escolha.

---
# 5.7 O que aprendemos nesta parte?

✔️ Validação cruzada é mais robusta que treino/teste  
✔️ K‑Fold é o método mais comum  
✔️ CV reduz variância da avaliação  
✔️ CV é essencial para comparar modelos  
✔️ CV é o padrão‑ouro em ML clássico  

Agora estamos prontos para o próximo passo:

**Regularização — Ridge e Lasso (a solução para overfitting).**

<details>
<summary>🟠 Explicação avançada — CV como estimador quase não viesado</summary>

A validação cruzada é um estimador quase não viesado do erro esperado:

$$
E[(Y - \hat{f}(X))^2]
$$

Ela aproxima o erro real porque:

- cada fold é uma amostra independente  
- cada fold é usado como teste exatamente uma vez  

Isso reduz o viés e a variância da estimativa.

</details>

<a id="regularizacao"></a>
# 6. Regularização — Ridge e Lasso

No módulo anterior, vimos que:

- modelos muito complexos → **overfitting**  
- modelos muito simples → **underfitting**  

Agora vamos aprender a técnica mais poderosa para controlar complexidade:

> **Regularização**  

A ideia é simples:

> Penalizar modelos com coeficientes muito grandes.

Isso reduz variância, melhora generalização e evita overfitting.

---
# 6.1 Por que regularização funciona?

Em regressão linear tradicional, minimizamos:

$$
\sum (y_i - \hat{y}_i)^2
$$

Na regularização, adicionamos uma penalidade:

$$
\sum (y_i - \hat{y}_i)^2 + \lambda \cdot \text{penalidade}
$$

Onde:
- **λ** controla a força da penalização  
- λ = 0 → regressão normal  
- λ grande → coeficientes encolhem  

---
# 6.2 Ridge Regression (L2)

Penalidade:

$$
\lambda \sum \beta_j^2
$$

Consequências:
- encolhe coeficientes  
- nunca zera coeficientes  
- ótimo para multicolinearidade  

Vamos aplicar Ridge para prever `eval` usando várias variáveis.

In [ ]:
X = ratings_df[["beauty", "age", "students"]]
y = ratings_df["eval"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train)

ridge.coef_, ridge.intercept_

🟣 **Interpretação**

- coeficientes menores que os da regressão normal  
- modelo mais estável  
- menos sensível a ruído  

---
# 6.3 Avaliando Ridge

In [ ]:
y_pred_ridge = ridge.predict(X_test)

rmse_ridge = mean_squared_error(y_test, y_pred_ridge, squared=False)
rmse_ridge

🟣 **Interpretação**

- Se RMSE_ridge < RMSE_linear → regularização ajudou  
- Se RMSE_ridge > RMSE_linear → λ muito alto  

---
# 6.4 Lasso Regression (L1)

Penalidade:

$$
\lambda \sum |\beta_j|
$$

Consequências:
- encolhe coeficientes  
- **zera coeficientes** (seleção automática de variáveis)  
- ótimo para feature selection  

Vamos aplicar Lasso.

In [ ]:
lasso = Lasso(alpha=0.05)
lasso.fit(X_train, y_train)

lasso.coef_, lasso.intercept_

🟣 **Interpretação**

- coeficientes podem virar zero  
- modelo mais simples  
- útil quando há muitas variáveis  

---
# 6.5 Avaliando Lasso

In [ ]:
y_pred_lasso = lasso.predict(X_test)

rmse_lasso = mean_squared_error(y_test, y_pred_lasso, squared=False)
rmse_lasso

🟣 **Interpretação**

- Lasso pode melhorar generalização  
- mas pode remover variáveis importantes se λ for alto demais  

---
# 6.6 Comparando Linear vs Ridge vs Lasso

In [ ]:
model_linear = LinearRegression()
model_linear.fit(X_train, y_train)
rmse_linear = mean_squared_error(y_test, model_linear.predict(X_test), squared=False)

rmse_linear, rmse_ridge, rmse_lasso

🟣 **Conclusão**

- Linear → sem penalização  
- Ridge → coeficientes menores, mais estável  
- Lasso → pode zerar coeficientes  

Regularização é essencial quando:
- há muitas variáveis  
- há multicolinearidade  
- há risco de overfitting  

---
# 6.7 Visualização dos coeficientes

In [ ]:
coef_df = pd.DataFrame({
    "Variável": X.columns,
    "Linear": model_linear.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
})

coef_df

🟣 **Interpretação**

- Ridge encolhe coeficientes suavemente  
- Lasso encolhe e pode zerar  
- Linear não encolhe nada  

---
# 6.8 O que aprendemos nesta parte?

✔️ Regularização controla complexidade  
✔️ Ridge (L2) encolhe coeficientes  
✔️ Lasso (L1) encolhe e zera coeficientes  
✔️ Regularização reduz overfitting  
✔️ Regularização melhora generalização  
✔️ Regularização é essencial em ML moderno  

Agora estamos prontos para:

**Seleção de Modelos — como escolher o melhor modelo preditivo.**

<details>
<summary>🟠 Explicação avançada — Por que Lasso zera coeficientes?</summary>

A penalidade L1 cria uma região de solução com “quinas”.

Geometricamente:

- L2 → círculo  
- L1 → losango  

As quinas do losango fazem com que a solução ótima caia exatamente em zero.

Por isso Lasso faz seleção de variáveis.

</details>

<a id="selecao"></a>
# 7. Seleção de Modelos — Comparando Modelos de Forma Justa

Agora que já aprendemos:

- treino/teste  
- métricas preditivas  
- overfitting/underfitting  
- validação cruzada  
- regularização (Ridge e Lasso)  

Estamos prontos para responder a pergunta central:

> **Qual modelo é o melhor?**

Para isso, precisamos comparar modelos de forma justa e robusta.

---
# 7.1 Por que comparar modelos?

Porque modelos diferentes têm:

- complexidades diferentes  
- vieses diferentes  
- variâncias diferentes  
- sensibilidades diferentes  

E o melhor modelo é aquele que:

> **generaliza melhor para novos dados.**

Não é o modelo com menor erro no treino.  
Não é o modelo com maior R² no treino.  

É o modelo com **menor erro preditivo**.

---
# 7.2 Comparando Linear vs Ridge vs Lasso com validação cruzada

Vamos usar K‑Fold CV para comparar:

- Regressão Linear  
- Ridge  
- Lasso  

Usaremos RMSE como métrica.

In [ ]:
from sklearn.pipeline import make_pipeline

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=0.05)
}

results = {}

for name, model in models.items():
    scores = cross_val_score(
        model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=5
    )
    results[name] = -scores.mean()

results

🟣 **Interpretação**

- Quanto menor o RMSE, melhor o modelo.  
- A comparação é justa porque todos os modelos foram avaliados nos mesmos folds.  

---
# 7.3 Tabela comparativa

In [ ]:
pd.DataFrame.from_dict(results, orient="index", columns=["RMSE_CV"])

🟣 **Interpretação**

- O modelo com menor RMSE_CV é o melhor preditor.  
- Diferenças pequenas podem não ser significativas.  
- Diferenças grandes indicam vantagem clara.  

---
# 7.4 Visualização da comparação

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results.keys(), results.values(), color=["blue", "green", "red"])
plt.ylabel("RMSE (Cross-Validation)")
plt.title("Comparação de Modelos — RMSE CV")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Visualmente, fica fácil ver qual modelo é melhor.  
- Ridge e Lasso tendem a melhorar estabilidade.  

---
# 7.5 Seleção de hiperparâmetros (α) para Ridge e Lasso

O desempenho depende de **α**:

- α pequeno → pouca regularização  
- α grande → muita regularização  

Vamos testar vários valores.

In [ ]:
alphas = np.logspace(-3, 3, 20)

ridge_scores = []
lasso_scores = []

for a in alphas:
    ridge_model = Ridge(alpha=a)
    lasso_model = Lasso(alpha=a, max_iter=5000)

    ridge_rmse = -cross_val_score(
        ridge_model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=5
    ).mean()

    lasso_rmse = -cross_val_score(
        lasso_model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=5
    ).mean()

    ridge_scores.append(ridge_rmse)
    lasso_scores.append(lasso_rmse)

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogx(alphas, ridge_scores, label="Ridge")
plt.semilogx(alphas, lasso_scores, label="Lasso")
plt.xlabel("alpha (log scale)")
plt.ylabel("RMSE CV")
plt.title("Seleção de Hiperparâmetros — Ridge e Lasso")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Cada curva tem um ponto ótimo.  
- α muito pequeno → overfitting  
- α muito grande → underfitting  

O ponto mínimo da curva é o melhor α.

---
# 7.6 Seleção automática do melhor modelo

Vamos escolher o modelo com menor RMSE CV.

In [ ]:
best_model = min(results, key=results.get)
best_model

🟣 **Conclusão**

- Este é o modelo que generaliza melhor.  
- Ele será usado no **Projeto Final**.  

---
# 7.7 O que aprendemos nesta parte?

✔️ Como comparar modelos de forma justa  
✔️ Como usar validação cruzada para avaliação robusta  
✔️ Como comparar Linear, Ridge e Lasso  
✔️ Como escolher hiperparâmetros (α)  
✔️ Como selecionar o melhor modelo preditivo  

Agora estamos prontos para:

**PARTE 8 — Projeto Final: Modelo Preditivo Completo.**

<details>
<summary>🟠 Explicação avançada — Por que CV é essencial para seleção de modelos?</summary>

Porque comparar modelos usando apenas treino/teste:

- tem alta variância  
- depende da divisão  
- pode favorecer modelos mais complexos  

A validação cruzada:

- usa todos os dados  
- reduz variância  
- é estatisticamente mais estável  
- evita overfitting na seleção de modelos  

Por isso é o padrão‑ouro em ML clássico.

</details>

<a id="projeto"></a>
# 8. Projeto Final — Construindo um Modelo Preditivo Completo

Neste projeto, vamos aplicar **todo o pipeline de Machine Learning clássico**:

1. Seleção de variáveis  
2. Divisão treino/teste  
3. Ajuste de modelos (Linear, Ridge, Lasso)  
4. Métricas preditivas  
5. Validação cruzada  
6. Seleção de hiperparâmetros  
7. Escolha do melhor modelo  
8. Interpretação  
9. Visualização  

Usaremos o dataset *teaching ratings* e tentaremos prever:

> **eval** (avaliação do professor)

usando múltiplas variáveis explicativas.

---
# 8.1 Seleção das variáveis

Vamos usar:

- beauty  
- age  
- students  
- female_dummy  

Essas variáveis representam:
- aparência  
- idade  
- tamanho da turma  
- gênero  

In [ ]:
X = ratings_df[["beauty", "age", "students", "female_dummy"]]
y = ratings_df["eval"]

X.head()

---
# 8.2 Divisão treino/teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

len(X_train), len(X_test)

---
# 8.3 Ajustando os modelos

Vamos ajustar:

- Regressão Linear  
- Ridge  
- Lasso  

In [ ]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=0.05, max_iter=5000)
}

fitted = {name: model.fit(X_train, y_train) for name, model in models.items()}

fitted

---
# 8.4 Avaliação no conjunto de teste

In [ ]:
results_test = {}

for name, model in fitted.items():
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results_test[name] = (rmse, mae, r2)

pd.DataFrame(results_test, index=["RMSE", "MAE", "R2"]).T

🟣 **Interpretação**

- O melhor modelo é o que tem **menor RMSE** e **maior R²** no teste.  
- Diferenças pequenas podem não ser significativas.  

---
# 8.5 Avaliação com validação cruzada

Agora vamos usar K‑Fold CV (k=5).

In [ ]:
results_cv = {}

for name, model in models.items():
    scores = cross_val_score(
        model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=5
    )
    results_cv[name] = -scores.mean()

pd.DataFrame.from_dict(results_cv, orient="index", columns=["RMSE_CV"])

🟣 **Interpretação**

- RMSE_CV é mais confiável que RMSE do teste.  
- O modelo com menor RMSE_CV é o melhor preditor.  

---
# 8.6 Seleção de hiperparâmetros (α) para Ridge e Lasso

In [ ]:
alphas = np.logspace(-3, 3, 20)

ridge_scores = []
lasso_scores = []

for a in alphas:
    ridge_model = Ridge(alpha=a)
    lasso_model = Lasso(alpha=a, max_iter=5000)

    ridge_rmse = -cross_val_score(
        ridge_model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=5
    ).mean()

    lasso_rmse = -cross_val_score(
        lasso_model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=5
    ).mean()

    ridge_scores.append(ridge_rmse)
    lasso_scores.append(lasso_rmse)

plt.figure(figsize=(8, 5))
plt.semilogx(alphas, ridge_scores, label="Ridge")
plt.semilogx(alphas, lasso_scores, label="Lasso")
plt.xlabel("alpha (log scale)")
plt.ylabel("RMSE CV")
plt.title("Seleção de Hiperparâmetros — Ridge e Lasso")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Cada curva tem um ponto mínimo → melhor α.  
- α pequeno → overfitting  
- α grande → underfitting  

---
# 8.7 Escolhendo o melhor modelo

Vamos selecionar o modelo com menor RMSE CV.

In [ ]:
best_model_name = min(results_cv, key=results_cv.get)
best_model_name

🟣 **Conclusão**

Este é o modelo que generaliza melhor para novos dados.

---
# 8.8 Interpretando os coeficientes do melhor modelo

In [ ]:
best_model = models[best_model_name].fit(X, y)

coef_df = pd.DataFrame({
    "Variável": X.columns,
    "Coeficiente": best_model.coef_
})

coef_df

🟣 **Interpretação**

- coeficientes positivos → aumentam eval  
- coeficientes negativos → reduzem eval  
- magnitude → força do efeito  

Regularização tende a encolher coeficientes.

---
# 8.9 Visualização: valores reais vs previstos

In [ ]:
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(7, 4))
plt.scatter(y_test, y_pred_best, alpha=0.7)
plt.xlabel("Valores reais")
plt.ylabel("Valores previstos")
plt.title(f"Melhor modelo: {best_model_name}")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Quanto mais próximos da diagonal, melhor.  
- Dispersão indica erro preditivo.  

---
# 8.10 Conclusão do Projeto Final

✔️ Construímos um pipeline completo de ML  
✔️ Comparámos Linear, Ridge e Lasso  
✔️ Usámos treino/teste e validação cruzada  
✔️ Selecionámos hiperparâmetros  
✔️ Escolhemos o melhor modelo  
✔️ Interpretámos coeficientes  
✔️ Avaliámos generalização  

Agora você domina **todo o ciclo de Machine Learning clássico**.

<details>
<summary>🟠 Reflexão avançada — Por que este pipeline é padrão em ML?</summary>

Porque ele:

- reduz viés  
- reduz variância  
- evita overfitting  
- permite comparação justa  
- funciona para qualquer modelo  
- é usado em produção  

Este pipeline é a base de:
- regressão logística  
- árvores  
- random forest  
- gradient boosting  
- redes neurais  

Você agora domina o núcleo do ML supervisionado.

</details>

<a id="apendice"></a>
# 9. Apêndice Matemático Avançado — Fundamentos de Generalização e Regularização

Este apêndice aprofunda os conceitos matemáticos que sustentam:

- treino/teste  
- validação cruzada  
- overfitting e underfitting  
- regularização (Ridge e Lasso)  
- decomposição viés‑variância  
- erro esperado  
- penalização L1 e L2  

Embora opcionais, esses fundamentos são essenciais para dominar ML.

---
# 9.1 Erro esperado e generalização

O objetivo de um modelo preditivo é minimizar:

$$
E[(Y - \hat{f}(X))^2]
$$

Mas não conhecemos a distribuição real de $(X, Y)$.

Por isso usamos:

- treino → aproximação da função  
- teste → aproximação do erro esperado  

A validação cruzada melhora essa estimativa.

---
# 9.2 Decomposição viés‑variância

O erro esperado pode ser decomposto em:

$$
E[(Y - \hat{f}(X))^2] = \text{Viés}^2 + \text{Variância} + \text{Ruído}
$$

Onde:

- **Viés** → erro por modelo simples demais  
- **Variância** → sensibilidade ao conjunto de treino  
- **Ruído** → parte imprevisível  

Overfitting = variância alta  
Underfitting = viés alto  

<details>
<summary>🟠 Demonstração — decomposição viés‑variância</summary>

Começamos com:

$$
E[(Y - \hat{f})^2]
$$

Substituímos $Y = f(X) + \varepsilon$:

$$
E[(f + \varepsilon - \hat{f})^2]
$$

Expandindo:

$$
E[(f - \hat{f})^2] + E[\varepsilon^2]
$$

Agora decompondo o primeiro termo:

$$
(f - E[\hat{f}])^2 + E[(\hat{f} - E[\hat{f}])^2]
$$

Que são:

- viés²  
- variância  
- ruído  

</details>

---
# 9.3 Regularização como controle de variância

Em regressão linear:

$$
\hat{\beta} = (X^\top X)^{-1} X^\top Y
$$

Se $X^\top X$ é mal condicionado (multicolinearidade), a variância explode.

Regularização corrige isso.

---
# 9.4 Ridge Regression (L2)

Ridge minimiza:

$$
\sum (y_i - \hat{y}_i)^2 + \lambda \sum \beta_j^2
$$

Solução fechada:

$$
\hat{\beta}_{ridge} = (X^\top X + \lambda I)^{-1} X^\top Y
$$

Consequências:
- reduz variância  
- encolhe coeficientes  
- nunca zera coeficientes  

<details>
<summary>🟠 Demonstração — solução de Ridge</summary>

Minimizamos:

$$
S(\beta) = (Y - X\beta)^\top(Y - X\beta) + \lambda \beta^\top \beta
$$

Derivando:

$$
-2X^\top(Y - X\beta) + 2\lambda \beta = 0
$$

Rearranjando:

$$
(X^\top X + \lambda I)\beta = X^\top Y
$$

Solução:

$$
\hat{\beta} = (X^\top X + \lambda I)^{-1} X^\top Y
$$

</details>

---
# 9.5 Lasso Regression (L1)

Lasso minimiza:

$$
\sum (y_i - \hat{y}_i)^2 + \lambda \sum |\beta_j|
$$

Não existe solução fechada.

Propriedades:
- encolhe coeficientes  
- pode zerar coeficientes  
- faz seleção de variáveis  

<details>
<summary>🟠 Por que Lasso zera coeficientes?</summary>

A penalidade L1 cria uma região de solução com quinas.

Geometricamente:

- L2 → círculo  
- L1 → losango  

As quinas do losango fazem com que a solução ótima caia exatamente em zero.

</details>

---
# 9.6 Validação cruzada como estimador do erro esperado

A validação cruzada estima:

$$
E[(Y - \hat{f}(X))^2]
$$

usando múltiplas divisões independentes.

Isso reduz variância e melhora estabilidade.

---
# 9.7 Seleção de hiperparâmetros

O hiperparâmetro λ controla:

- complexidade  
- variância  
- viés  

A escolha ótima é:

$$
\lambda^\* = \arg\min_\lambda RMSE_{CV}(\lambda)
$$

---
# 9.8 Interpretação geométrica da regularização

A solução da regressão linear é a projeção de Y no subespaço de X.

Regularização:

- altera o subespaço  
- encolhe a projeção  
- reduz sensibilidade a ruído  

---
# 9.9 Conexão com Machine Learning moderno

Regularização é usada em:

- regressão linear  
- regressão logística  
- SVM  
- árvores (poda)  
- random forest (bagging reduz variância)  
- gradient boosting (shrinkage)  
- redes neurais (L2, dropout, batch norm)  

Ou seja:

> **Regularização é o coração do ML moderno.**

---
# 9.10 Resumo do Apêndice

✔️ Erro esperado = viés² + variância + ruído  
✔️ Overfitting = variância alta  
✔️ Underfitting = viés alto  
✔️ Ridge reduz variância  
✔️ Lasso faz seleção de variáveis  
✔️ Validação cruzada estima erro real  
✔️ λ controla complexidade  
✔️ Regularização é essencial em ML  

Com isso, você domina a matemática por trás do Módulo 7.